In [1]:
import sqlite3
import pandas as pd

In [2]:
connection = sqlite3.connect("../outputs/olist.db")

In [3]:
cursor = connection.cursor()

In [4]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
sonuc = cursor.fetchall()
sonuc

[('etl_sonuc',),
 ('orders',),
 ('order_items',),
 ('customers',),
 ('products',),
 ('payments',),
 ('reviews',),
 ('translation',)]

In [5]:
orders_df = pd.read_csv("../data/olist_orders_dataset.csv")
order_items_df = pd.read_csv("../data/olist_order_items_dataset.csv")
customers_df = pd.read_csv("../data/olist_customers_dataset.csv")
products_df = pd.read_csv("../data/olist_products_dataset.csv")
payments_df = pd.read_csv("../data/olist_order_payments_dataset.csv")
reviews_df = pd.read_csv("../data/olist_order_reviews_dataset.csv")

In [6]:
tum_dfler = {
    "orders": orders_df,
    "order_items": order_items_df,
    "customers": customers_df,
    "products": products_df,
    "payments": payments_df,
    "reviews": reviews_df
}

In [7]:
for isim, df in tum_dfler.items():
    print(f"{isim}: {df.shape}")

orders: (99441, 8)
order_items: (112650, 7)
customers: (99441, 5)
products: (32951, 9)
payments: (103886, 5)
reviews: (99224, 7)


In [8]:
for isim, df in tum_dfler.items():
    df.to_sql(isim, connection, if_exists="replace", index=False)

In [9]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", connection)

,name
0,etl_sonuc
1,translation
2,orders
3,order_items
4,customers
5,products
6,payments
7,reviews


In [10]:
pd.read_sql_query("PRAGMA table_info(payments);", connection)

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,payment_sequential,INTEGER,0,None,0
2,2,payment_type,TEXT,0,None,0
3,3,payment_installments,INTEGER,0,None,0
4,4,payment_value,REAL,0,None,0


* **payments** tablosundan, ödeme değeri (`payment_value`) **500'den büyük** olan kayıtları, değeri **büyükten küçüğe** sıralı şekilde **ilk 10** tanesini getir.

In [11]:
sonuc_df = pd.read_sql_query("""
    SELECT order_id, payment_type, payment_value
    FROM payments
    WHERE payment_value > 500
    ORDER BY payment_value DESC
    LIMIT 10;
""", connection)
sonuc_df

,order_id,payment_type,payment_value
0,03caa2c082116e1d31e67e9ae3700499,credit_card,13664.08
1,736e1922ae60d0d6a89247b851902527,boleto,7274.88
2,0812eb902a67711a1cb742b3cdaa65ae,credit_card,6929.31
3,fefacc66af859508bf1a7934eab1e97f,boleto,6922.21
4,f5136e38d1a14a4dbd87dff67da82701,boleto,6726.66
5,2cc9089445046817a7539d90805e6e5a,boleto,6081.54
6,a96610ab360d42a2e5335a3998b4718a,credit_card,4950.34
7,b4c4b76c642808cbe472a32b86cddc95,credit_card,4809.44
8,199af31afc78c699f0dbf71fb178d4d4,credit_card,4764.34
9,8dbc85d1447242f3b127dda390d56e19,credit_card,4681.78


* **payments** tablosunda, her `payment_type` için: kaç tane ödeme yapıldığı (`COUNT`) ve toplam ödeme değeri (`SUM`) nedir?

In [12]:
odeme_dagilimi_df = pd.read_sql_query("""
    SELECT payment_type, COUNT(*) AS islem_sayisi, SUM(payment_value) AS toplam_deger
    FROM payments
    GROUP BY payment_type
    ORDER BY toplam_deger DESC;
""", connection)
odeme_dagilimi_df

,payment_type,islem_sayisi,toplam_deger
0,credit_card,76795,1.254208e+07
1,boleto,19784,2.869361e+06
2,voucher,5775,3.794369e+05
3,debit_card,1529,2.179898e+05
4,not_defined,3,0.000000e+00


### Ödeme Yöntemi Dağılımı 

**İş sorusu:** Hangi ödeme yöntemleri en yüksek işlem sayısına ve toplam gelire sahip?

- **Credit card** açık ara en baskın ödeme yöntemi: 76.795 işlem, ~12.5M toplam değer — işlem sayısının %74'ü, toplam gelirin ~%78'ini oluşturuyor.
- **Boleto** (Brezilya'ya özgü bir tür banka havalesi/fatura ödeme yöntemi) ikinci sırada, ancak credit card'ın çok gerisinde kalıyor.
- `not_defined` olarak etiketlenmiş 3 satır var ve bunların değeri 0.0 — bu muhtemelen bir veri kalitesi sorununu (eksik/bozuk kayıt) işaret ediyor. Satır sayısı çok az olduğu için genel analizi etkilemiyor, ama sonuç/özet bölümünde veri setindeki bu tür küçük tutarsızlıklara değinmek gerekiyor.

### JOIN Pratiği: Siparişler ve Müşteri Şehirleri

In [13]:
pd.read_sql_query("""
    SELECT o.order_id, o.order_status, c.customer_city, c.customer_state
    FROM orders AS o
    INNER JOIN customers AS c ON o.customer_id = c.customer_id
    LIMIT 10;
""", connection)

,order_id,order_status,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,santo andre,SP
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,congonhinhas,PR
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,santa rosa,RS
7,6514b8ad8028c9f2cc2374ded245783f,delivered,nilopolis,RJ
8,76c6e866289321a7c93b82b54852dc33,delivered,faxinalzinho,RS
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,sorocaba,SP


### İş Sorusu 1: Aylık Toplam Gelir Trendi

**Soru:** Aylık toplam gelir trendi nasıl değişiyor?

**Yöntem:** `orders` ve `order_items` tabloları `order_id` üzerinden INNER JOIN ile birleştirildi. `strftime('%Y-%m', ...)` ile sipariş tarihinden yıl-ay bilgisi çıkarılıp GROUP BY ile aylara göre gruplandı. Sipariş sayısı için `COUNT(DISTINCT order_id)` kullanıldı, çünkü bir siparişte birden fazla ürün olabileceğinden JOIN sonrası aynı sipariş birden fazla satırda tekrar edebilir.

In [14]:
aylik_gelir_df = pd.read_sql_query("""
    SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS ay,
           SUM(oi.price) AS toplam_gelir,
           COUNT(DISTINCT o.order_id) AS siparis_sayisi
    FROM orders AS o
    INNER JOIN order_items AS oi ON o.order_id = oi.order_id
    GROUP BY ay
    ORDER BY ay;
""", connection)
aylik_gelir_df

,ay,toplam_gelir,siparis_sayisi
0,2016-09,267.36,3
1,2016-10,49507.66,308
2,2016-12,10.90,1
3,2017-01,120312.87,789
4,2017-02,247303.02,1733
5,2017-03,374344.30,2641
6,2017-04,359927.23,2391
7,2017-05,506071.14,3660
8,2017-06,433038.60,3217
9,2017-07,498031.48,3969


**Bulgular:**
- 2017 boyunca istikrarlı bir büyüme trendi var; Kasım 2017'de belirgin bir sıçrama gözlemleniyor (kampanya dönemine denk gelebilir).
- 2018 boyunca gelir yüksek ve nispeten stabil seyrediyor.
- 2016-09/2016-10/2016-12 ve 2018-09 dönemlerinde anormal derecede düşük değerler var — bu gerçek bir iş sorunu değil, veri setinin bu tarihlerde henüz başlamamış/kesilmiş olmasından kaynaklanıyor. Analiz ve yorumlarda bu dönemler dikkate alınmamalı.

### İş Sorusu 2: En Çok Gelir Getiren İlk 10 Kategori

**Soru:** Hangi ürün kategorileri en yüksek toplam geliri getiriyor?

**Yöntem:** `order_items`, `products` ve `product_category_name_translation` tabloları zincirleme (çift) INNER JOIN ile birleştirildi — önce `product_id` üzerinden order_items↔products, sonra `product_category_name` üzerinden products↔translation. Bu sayede Portekizce kategori isimleri İngilizce'ye çevrilerek gruplandı.

In [15]:
translation_df = pd.read_csv("../data/product_category_name_translation.csv")
translation_df.shape

(71, 2)

In [16]:
translation_df.to_sql("translation", connection, if_exists="replace", index=False)

71

In [17]:
pd.read_sql_query("SELECT * FROM translation LIMIT 5;", connection)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [18]:
kategori_gelir_df = pd.read_sql_query("""
    SELECT t.product_category_name_english AS kategori,
           SUM(oi.price) AS toplam_gelir,
           COUNT(*) AS satis_adedi
    FROM order_items AS oi
    INNER JOIN products AS p ON oi.product_id = p.product_id
    INNER JOIN translation AS t ON p.product_category_name = t.product_category_name
    GROUP BY kategori
    ORDER BY toplam_gelir DESC
    LIMIT 10;
""", connection)
kategori_gelir_df

,kategori,toplam_gelir,satis_adedi
0,health_beauty,1258681.34,9670
1,watches_gifts,1205005.68,5991
2,bed_bath_table,1036988.68,11115
3,sports_leisure,988048.97,8641
4,computers_accessories,911954.32,7827
5,furniture_decor,729762.49,8334
6,cool_stuff,635290.85,3796
7,housewares,632248.66,6964
8,auto,592720.11,4235
9,garden_tools,485256.46,4347


**Bulgular:**
- `health_beauty` en yüksek toplam gelire sahip (1.26M), ancak `watches_gifts` çok yakın ikinci (1.2M) — üstelik çok daha az satış adediyle (5991 vs 9670). Bu, watches_gifts kategorisinin ortalama birim fiyatının belirgin şekilde daha yüksek olduğunu gösteriyor.
- `bed_bath_table` en yüksek satış adedine sahip (11115) ama gelirde 3. sırada — düşük birim fiyatlı, yüksek hacimli bir kategori.
- Bu gözlem, yüksek birim fiyatlı kategorilere yönelik hedefli pazarlama fırsatı olarak değerlendirilebilir.

### İş Sorusu 3: Eyalet Bazlı Ortalama Teslimat Süresi

**Soru:** Ortalama teslimat süresi eyaletlere göre nasıl farklılaşıyor?

**Yöntem:** `orders` ve `customers` tabloları `customer_id` üzerinden INNER JOIN ile birleştirildi. `julianday()` fonksiyonu ile teslimat tarihi ve sipariş tarihi arasındaki fark gün cinsinden hesaplandı, eyalete göre gruplanıp ortalaması alındı. Henüz teslim edilmemiş (tarih bilgisi NULL olan) siparişler analiz dışında bırakıldı.

In [19]:
sehir_teslimat_df = pd.read_sql_query("""
    SELECT c.customer_state,
           ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)), 1) AS ortalama_teslimat_gun,
           COUNT(*) AS siparis_sayisi
    FROM orders AS o
    INNER JOIN customers AS c ON o.customer_id = c.customer_id
    WHERE o.order_delivered_customer_date IS NOT NULL
    GROUP BY c.customer_state
    ORDER BY ortalama_teslimat_gun DESC
    LIMIT 10;
""", connection)
sehir_teslimat_df

,customer_state,ortalama_teslimat_gun,siparis_sayisi
0,RR,29.4,41
1,AP,27.2,67
2,AM,26.4,145
3,AL,24.5,397
4,PA,23.8,946
5,MA,21.6,717
6,SE,21.5,335
7,CE,21.3,1279
8,AC,21.0,80
9,PB,20.4,517


**Bulgular:**
- En uzun ortalama teslimat süreleri Brezilya'nın kuzey/Amazon bölgesindeki eyaletlerde (RR: 29.4 gün, AP: 27.2 gün, AM: 26.4 gün) — bu bölgelerin ana lojistik merkezlerden coğrafi uzaklığıyla örtüşüyor.
- Bu eyaletlerde sipariş sayısı da düşük (örn. RR sadece 41 sipariş), yani hem talep az hem de teslimat performansı zayıf.
- Bu bulgu, kuzey bölgeler için bölgesel lojistik/depo ortaklığı gibi bir iyileştirme önerisine zemin oluşturabilir.

### İş Sorusu 4: Memnuniyet Skoru – Teslimat Süresi İlişkisi

**Soru:** Müşteri memnuniyet skoru ile teslimat süresi arasında bir ilişki var mı?

**Yöntem:** `orders` ve `reviews` tabloları `order_id` üzerinden INNER JOIN ile birleştirildi. Her review_score (1-5) için ortalama teslimat süresi (julianday farkı) hesaplandı.

In [20]:
memnuniyet_teslimat_df = pd.read_sql_query("""
    SELECT r.review_score,
           ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)), 1) AS ortalama_teslimat_gun,
           COUNT(*) AS siparis_sayisi
    FROM orders AS o
    INNER JOIN reviews AS r ON o.order_id = r.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
    GROUP BY r.review_score
    ORDER BY r.review_score;
""", connection)
memnuniyet_teslimat_df

,review_score,ortalama_teslimat_gun,siparis_sayisi
0,1,21.3,9409
1,2,16.7,2941
2,3,14.3,7962
3,4,12.3,18987
4,5,10.7,57060


**Bulgular:**
- Net ve monoton bir ters ilişki var: review_score arttıkça ortalama teslimat süresi düzenli şekilde azalıyor (1 puan: 21.3 gün → 5 puan: 10.7 gün).
- En düşük puanlı siparişlerin teslimat süresi, en yüksek puanlılara göre neredeyse iki kat daha uzun.
- Bu bulgu, müşteri memnuniyetsizliğinin önemli bir kısmının teslimat gecikmesinden kaynaklandığına işaret ediyor — teslimat süresini kısaltmaya yönelik bir iyileştirme, memnuniyeti doğrudan artırabilir.

### İş Sorusu 5: Ödeme Yöntemi – Ortalama Sipariş Değeri İlişkisi

**Soru:** Hangi ödeme yöntemleri en yüksek ortalama sipariş değerine sahip?

**Yöntem:** `payments` ve `orders` tabloları `order_id` üzerinden INNER JOIN ile birleştirildi. Her ödeme yöntemi için ortalama ödeme tutarı ve işlem sayısı hesaplandı.

In [21]:
odeme_deger_df = pd.read_sql_query("""
    SELECT p.payment_type,
           ROUND(AVG(p.payment_value), 2) AS ortalama_odeme,
           COUNT(*) AS islem_sayisi
    FROM payments AS p
    INNER JOIN orders AS o ON p.order_id = o.order_id
    GROUP BY p.payment_type
    ORDER BY ortalama_odeme DESC;
""", connection)
odeme_deger_df

,payment_type,ortalama_odeme,islem_sayisi
0,credit_card,163.32,76795
1,boleto,145.03,19784
2,debit_card,142.57,1529
3,voucher,65.70,5775
4,not_defined,0.00,3


**Bulgular:**
- `credit_card` hem en yüksek işlem sayısına hem de en yüksek ortalama ödeme tutarına (163.32) sahip.
- `voucher` belirgin şekilde daha düşük ortalama değere sahip (65.70) — küçük promosyon/indirim amaçlı kullanıldığına işaret ediyor.
- `not_defined` kategorisi (3 işlem, 0.00 TL) daha önceki analizde de gözlemlenen bir veri kalitesi sorununu doğruluyor.

In [22]:
aylik_gelir_df.to_csv("../dashboard/data/aylik_gelir.csv", index=False)
kategori_gelir_df.to_csv("../dashboard/data/kategori_gelir.csv", index=False)
sehir_teslimat_df.to_csv("../dashboard/data/eyalet_teslimat.csv", index=False)
memnuniyet_teslimat_df.to_csv("../dashboard/data/memnuniyet_teslimat.csv", index=False)
odeme_deger_df.to_csv("../dashboard/data/odeme_deger.csv", index=False)

print("5 CSV dosyasi dashboard/data/ klasorune export edildi.")

5 CSV dosyasi dashboard/data/ klasorune export edildi.
